In [ ]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import os

# ── 1. Load all US ZIP codes ───────────────────────────────────────────────────
gaz = pd.read_csv(
    "../data/raw/usps/2024_gaz_zcta_national.txt",
    sep='\t'
)
gaz.columns = gaz.columns.str.strip()
all_zips = gaz['GEOID'].astype(str).str.zfill(5).tolist()
print(f"Total ZIP codes: {len(all_zips)}")

# ── 2. API functions ───────────────────────────────────────────────────────────
def get_usps_routes(zip_code):
    url = (
        f"https://gis.usps.com/arcgis/rest/services/EDDM/selectZIP/"
        f"GPServer/routes/execute"
        f"?f=json&env:outSR=4326&ZIP={zip_code}&Rte_Box=R&UserName=EDDM"
    )
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        return data['results'][0]['value']['features']
    except:
        return []

def parse_geometry(geom_data):
    try:
        if geom_data and 'paths' in geom_data:
            paths = geom_data['paths']
            if len(paths) == 1:
                return LineString(paths[0])
            elif len(paths) > 1:
                lines = [LineString(p) for p in paths if len(p) >= 2]
                return MultiLineString(lines)
    except:
        pass
    return None

def fetch_zip(zip_code):
    routes = get_usps_routes(zip_code)
    results = []
    for r in routes:
        attrs = r['attributes'].copy()
        attrs['geometry'] = parse_geometry(r.get('geometry'))
        results.append(attrs)
    return zip_code, results

# ── 3. Parallel collection in batches ─────────────────────────────────────────
all_routes  = []
failed_zips = []

MAX_WORKERS = 5
BATCH_SIZE  = 1000
BATCH_DELAY = 5  # seconds between batches

# split into batches
batches = [all_zips[i:i+BATCH_SIZE] for i in range(0, len(all_zips), BATCH_SIZE)]
print(f"Total batches: {len(batches)} | Workers: {MAX_WORKERS} | Delay: {BATCH_DELAY}s\n")

for batch_num, batch in enumerate(batches):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_zip, z): z for z in batch}
        
        for future in tqdm(as_completed(futures), total=len(batch),
                           desc=f"Batch {batch_num+1}/{len(batches)}"):
            zip_code = futures[future]
            try:
                _, routes = future.result()
                all_routes.extend(routes)
            except Exception as e:
                failed_zips.append(zip_code)

    print(f"  → Batch {batch_num+1} done | Routes so far: {len(all_routes)} | Failed: {len(failed_zips)}")

    # pause between batches (not after the last one)
    if batch_num < len(batches) - 1:
        time.sleep(BATCH_DELAY)

print(f"\nFinished!")
print(f"Total routes: {len(all_routes)}")
print(f"Failed ZIPs:  {len(failed_zips)}")

# ── 4. Convert to GeoDataFrame and save ───────────────────────────────────────
usps_gdf = gpd.GeoDataFrame(all_routes, geometry='geometry', crs='EPSG:4326')
usps_gdf = usps_gdf[usps_gdf.geometry.notna()]

out_path = "../data/processed/us_usps_routes.gpkg"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
usps_gdf.to_file(out_path, driver="GPKG")
print(f"Saved {len(usps_gdf)} routes to {out_path}")

# ── 5. Save failed ZIPs for retry ─────────────────────────────────────────────
if failed_zips:
    pd.Series(failed_zips).to_csv(
        "../data/raw/usps/failed_zips.csv", index=False
    )
    print(f"Failed ZIPs saved for retry")

In [ ]:
import pandas as pd
import geopandas as gpd

# load US USPS routes
usps_us = gpd.read_file("../data/processed/us_usps_routes.gpkg")

# load all ZIP codes
gaz = pd.read_csv("../data/raw/usps/2024_gaz_zcta_national.txt", sep='\t')
gaz.columns = gaz.columns.str.strip()
all_zips = gaz['GEOID'].astype(str).str.zfill(5).tolist()

# find ZIPs that returned 0 routes
collected_zips = set(usps_us['ZIP_CODE'].astype(str).str.zfill(5).tolist())
empty_zips = [z for z in all_zips if z not in collected_zips]

print(f"Total ZIPs: {len(all_zips)}")
print(f"ZIPs with routes: {len(collected_zips)}")
print(f"ZIPs that returned empty: {len(empty_zips)}")

# save for re-collection
pd.Series(empty_zips).to_csv("../data/raw/usps/empty_zips.csv", index=False)
print("Saved to empty_zips.csv")

In [ ]:
import pandas as pd

empty_zips = pd.read_csv("../data/raw/usps/empty_zips.csv", header=None)[0].astype(str).str.zfill(5).tolist()

# split into 3 equal parts
n = len(empty_zips) // 3
part1 = empty_zips[:n]           # ~7,013 ZIPs → Laptop 1
part2 = empty_zips[n:2*n]        # ~7,013 ZIPs → Laptop 2
part3 = empty_zips[2*n:]         # ~7,015 ZIPs → HPCC

pd.Series(part1).to_csv("../data/raw/usps/empty_zips_part1.csv", index=False)
pd.Series(part2).to_csv("../data/raw/usps/empty_zips_part2.csv", index=False)
pd.Series(part3).to_csv("../data/raw/usps/empty_zips_part3.csv", index=False)

print(f"Part 1: {len(part1)} ZIPs")
print(f"Part 2: {len(part2)} ZIPs")
print(f"Part 3: {len(part3)} ZIPs")

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import os

# ── CHANGE THIS ON EACH MACHINE ───────────────────────────────────────────────
PART       = 1          # 1, 2, or 3
MAX_WORKERS = 10
BATCH_SIZE  = 1000
BATCH_DELAY = 5

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE     = "C:/Users/julak/Documents/broadband_project"  # change on HPCC
ZIP_FILE = f"{BASE}/data/raw/usps/empty_zips_part{PART}.csv"
OUT_FILE = f"{BASE}/data/raw/usps/new_routes_part{PART}.gpkg"

# ── Load ZIPs ──────────────────────────────────────────────────────────────────
empty_zips = pd.read_csv(ZIP_FILE, header=None)[0].astype(str).str.zfill(5).tolist()
print(f"Part {PART}: {len(empty_zips)} ZIPs to process")

# ── API functions ──────────────────────────────────────────────────────────────
def get_usps_routes(zip_code):
    url = (
        f"https://gis.usps.com/arcgis/rest/services/EDDM/selectZIP/"
        f"GPServer/routes/execute"
        f"?f=json&env:outSR=4326&ZIP={zip_code}&Rte_Box=R&UserName=EDDM"
    )
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        return data['results'][0]['value']['features']
    except:
        return []

def parse_geometry(geom_data):
    try:
        if geom_data and 'paths' in geom_data:
            paths = geom_data['paths']
            if len(paths) == 1:
                return LineString(paths[0])
            elif len(paths) > 1:
                lines = [LineString(p) for p in paths if len(p) >= 2]
                return MultiLineString(lines)
    except:
        pass
    return None

def fetch_zip(zip_code):
    routes = get_usps_routes(zip_code)
    results = []
    for r in routes:
        attrs = r['attributes'].copy()
        attrs['geometry'] = parse_geometry(r.get('geometry'))
        results.append(attrs)
    return zip_code, results

# ── Run collection ─────────────────────────────────────────────────────────────
new_routes  = []
failed_zips = []

batches = [empty_zips[i:i+BATCH_SIZE] for i in range(0, len(empty_zips), BATCH_SIZE)]
print(f"Total batches: {len(batches)} | Workers: {MAX_WORKERS}\n")

for batch_num, batch in enumerate(batches):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_zip, z): z for z in batch}
        for future in tqdm(as_completed(futures), total=len(batch),
                           desc=f"Batch {batch_num+1}/{len(batches)}"):
            zip_code = futures[future]
            try:
                _, routes = future.result()
                new_routes.extend(routes)
            except Exception as e:
                failed_zips.append(zip_code)

    print(f"  → Batch {batch_num+1} done | Routes: {len(new_routes)} | Failed: {len(failed_zips)}")
    if batch_num < len(batches) - 1:
        time.sleep(BATCH_DELAY)

print(f"\nFinished! New routes: {len(new_routes)} | Failed: {len(failed_zips)}")

# ── Save results ───────────────────────────────────────────────────────────────
if len(new_routes) > 0:
    gdf = gpd.GeoDataFrame(new_routes, geometry='geometry', crs='EPSG:4326')
    gdf = gdf[gdf.geometry.notna()]
    gdf.to_file(OUT_FILE, driver="GPKG")
    print(f"Saved {len(gdf)} routes to {OUT_FILE}")
else:
    print("No new routes found for this part")

if failed_zips:
    pd.Series(failed_zips).to_csv(
        f"{BASE}/data/raw/usps/failed_zips_part{PART}.csv", index=False
    )

In [ ]:
import geopandas as gpd
import pandas as pd

BASE = "C:/Users/julak/Documents/broadband_project"

# ── Load all parts ─────────────────────────────────────────────────────────────
existing = gpd.read_file(f"{BASE}/data/processed/us_usps_routes.gpkg")
part1    = gpd.read_file(f"{BASE}/data/raw/usps/new_routes_part1.gpkg")
part2    = gpd.read_file(f"{BASE}/data/raw/usps/new_routes_part2.gpkg")
part3    = gpd.read_file(f"{BASE}/data/raw/usps/new_routes_part3.gpkg")

print(f"Existing routes: {len(existing)}")
print(f"Part 1 new routes: {len(part1)}")
print(f"Part 2 new routes: {len(part2)}")
print(f"Part 3 new routes: {len(part3)}")

# ── Drop FID column if exists ──────────────────────────────────────────────────
for df in [existing, part1, part2, part3]:
    if 'FID' in df.columns:
        df.drop(columns=['FID'], inplace=True)

# ── Combine all ────────────────────────────────────────────────────────────────
combined = pd.concat([existing, part1, part2, part3], ignore_index=True)
combined = gpd.GeoDataFrame(combined, geometry='geometry', crs='EPSG:4326')
combined = combined[combined.geometry.notna()]

print(f"\nTotal routes: {len(combined)}")
print(f"Unique ZIPs: {combined['ZIP_CODE'].nunique()}")

# ── Save ───────────────────────────────────────────────────────────────────────
out = f"{BASE}/data/processed/us_usps_routes_complete.gpkg"
combined.to_file(out, driver="GPKG")
print(f"Saved to {out}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"
viridis = cm.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(8, 6))

buffers = ['0m\n(Direct)', '100m\nBuffer', '200m\nBuffer']
covered = [106, 109, 112]
not_covered = [68, 65, 62]
total = 174

x = np.arange(len(buffers))
width = 0.5

bars1 = ax.bar(x, covered, width, label='USPS Route Overlaps', color=viridis(0.7))
bars2 = ax.bar(x, not_covered, width, bottom=covered, label='No USPS Route', color=viridis(0.2))

for bar, val in zip(bars1, covered):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height()/2 + bar.get_y(),
            f'{val}\n({val/total*100:.1f}%)', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

for bar, val in zip(bars2, not_covered):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height()/2 + bar.get_y(),
            f'{val}\n({val/total*100:.1f}%)', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

ax.set_xticks(x)
ax.set_xticklabels(buffers, fontsize=11)
ax.set_ylabel('Number of Challenge Hexagons', fontsize=12)
ax.set_title('Overlap of USPS Routes with\nFCC MAC Challenge Hexagons', fontsize=14, fontweight='bold')
ax.set_ylim(0, 190)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{out_dir}/rq3_usps_mac_overlap.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved!")